# Python 速成：看懂 Agent 相關程式碼核心基礎 

**目標**：30 分鐘內，看到課程程式碼裡的 `Iterator`、`_client`、`__init__`、`class X(BaseModel)`、`@tool`，能馬上說出「它在幹嘛」。

**怎麼讀這本**：每一節都是同一個節奏——

1. **一句白話**（先抓直覺）
2. **一張圖**（比喻）
3. **專有名詞**（面試、查文件時要講的正式說法）
4. **跑一格程式**（親眼看到結果）

**前置**：Python 3.11+；最後一節用到 `pydantic`（課程環境已裝）。不需要任何 API key，圖都已經生好放在 `images/`。

**完整版文件**：[abc-and-underscores.md](abc-and-underscores.md)、[oop-for-agent-code.md](oop-for-agent-code.md)

| 節 | 白話 | 專有名詞 |
|---|---|---|
| 1 | 類別是藍圖，物件是成品 | class、instance、`self`、`__init__` |
| 2 | 底線是門牌 | convention、name mangling、dunder |
| 3 | 認能力，不認身分 | abstract base class、duck typing |
| 4 | 書和書籤 | `Iterable`、`Iterator`、`iter()`／`next()` |
| 5 | 做好一道上一道 | generator、`yield`、lazy evaluation |
| 6 | 填空 vs 裝零件 | inheritance、composition、dependency injection |
| 7 | 誰會檢查資料？ | `dataclass`、`TypedDict`、Pydantic |
| 8 | 把函式送去加工 | decorator、registry |

---
## 1. 類別是藍圖，物件是成品

<img src="images/class_blueprint.jpg" alt="類別是藍圖，物件是成品" width="560">

**白話**：`class` 是一張設計圖，照著它可以做出很多台機器。每台機器自己記自己的數字，互不干擾。

**專有名詞**

| 白話 | 正式說法 |
|---|---|
| 設計圖 | **class**（類別） |
| 做出來的機器 | **object／instance**（物件／實例） |
| 出廠設定 | **`__init__`**（初始化方法，物件建好後自動執行） |
| 機器自己 | **`self`**（指向目前這個物件） |
| 機器記住的數字 | **attribute**（屬性） |
| 機器會做的動作 | **method**（方法） |

深度學習的對照：`nn.Linear` 是類別，`nn.Linear(768, 10)` 是物件。

In [ ]:
class Counter:
    def __init__(self, start):      # 出廠設定
        self.value = start          # 屬性：這台機器自己記住的數字

    def add(self):                  # 方法：這台機器會做的事
        self.value += 1
        return self.value


a = Counter(0)
b = Counter(100)
print("a.add() ->", a.add())
print("b.add() ->", b.add())
print("a 身上的屬性：", vars(a))
print("b 身上的屬性：", vars(b))

`a.add()` 其實就是 `Counter.add(a)`——Python 自動把 `a` 塞進 `self`。所以定義時要寫 `self`，呼叫時不用傳：

In [ ]:
print(Counter.add(a))      # 跟 a.add() 完全一樣
print(type(a), isinstance(a, Counter))

---
## 2. 底線是門牌

<img src="images/underscore_doors.jpg" alt="底線命名：四扇門" width="560">

**白話**：名字前後的底線數量，就是貼在門上的告示。

| 寫法 | 白話 | 正式說法 | Python 會強制嗎 |
|---|---|---|---|
| `name` | 大門，歡迎使用 | public | — |
| `_name` | 「員工專用」，門沒鎖 | **internal／non-public（約定）** | 否 |
| `__name` | 門牌被自動改寫 | **name mangling**（名稱改編） | 會改名 |
| `__name__` | Python 的機房 | **dunder**（double underscore）／special method | Python 自動呼叫 |

課程例子：`course_utils/llm.py` 的 `_endpoints()`、`_client()` 是內部零件；你只該用 `chat`、`stream`、`embed`。

### 2-1 `_name`：只是約定，門沒鎖

In [ ]:
class Client:
    def __init__(self):
        self._api_base = "http://localhost:11434"   # 內部用


c = Client()
print(c._api_base)     # 照樣讀得到——Python 不擋，靠的是大家守規矩

### 2-2 `__name`：自動改名，避免子類別撞名

這不是安全機制，是**防撞名**：父類別和子類別都取了 `__token`，Python 會分別改成 `_Parent__token`、`_Child__token`，兩個並存。

In [ ]:
class Parent:
    def __init__(self):
        self.__token = "parent"


class Child(Parent):
    def __init__(self):
        super().__init__()
        self.__token = "child"


c = Child()
print(vars(c))               # 兩個 token 都還在，名字被改過了
print(c._Parent__token)      # 知道改後的名字一樣拿得到

### 2-3 `__xxx__`：你寫、Python 叫

dunder 方法是**和 Python 語法掛鉤的介面**。你寫 `len(d)`，Python 實際去呼叫 `d.__len__()`：

In [ ]:
class Playlist:
    def __init__(self, songs):
        self.songs = list(songs)

    def __len__(self):                # len(p) 觸發
        return len(self.songs)

    def __getitem__(self, i):         # p[i] 觸發
        return self.songs[i]

    def __repr__(self):               # 在 notebook 顯示時觸發
        return f"Playlist({len(self)} 首)"


p = Playlist(["A", "B", "C"])
print(len(p), p[0], p)
print("B" in p)       # 沒寫 __contains__，Python 退而用 __getitem__ 一個個找

---
## 3. 認能力，不認身分：`collections.abc`

<img src="images/abc_usb_spec.jpg" alt="collections.abc：認能力，不認身分" width="560">

**白話**：充電器不管插進來的是手機還是耳機，只問「你是不是 USB-C 插孔」。函式也一樣：`sum()` 不在乎你給 list、tuple 還是 generator，只要**能用 `for` 一個個拿出來**就行。

**專有名詞**

- **ABC（Abstract Base Class，抽象基底類別）**：只寫「必須具備哪些能力」的規格。
- **`collections.abc`**：Python 內建的一整套容器規格（`Iterable`、`Sequence`、`Mapping`、`Callable`…）。
- **duck typing**：「走路像鴨子、叫聲像鴨子，就當它是鴨子」——有對應方法就能用。
- **型別標註（type hint）**：`nums: Iterable[int]` 是給人和 Pylance/mypy 看的，**執行時不檢查**。

In [ ]:
from collections.abc import Callable, Iterable, Iterator, Mapping, Sequence

samples = {
    "list": [1, 2, 3],
    "tuple": (1, 2, 3),
    "dict": {"a": 1},
    "generator": (n for n in range(3)),
    "int": 42,
    "function": print,
}
specs = [Iterable, Iterator, Sequence, Mapping, Callable]

print(f"{'':<10}" + "".join(f"{s.__name__:<10}" for s in specs))
for name, obj in samples.items():
    row = "".join(f"{'✓' if isinstance(obj, s) else '·':<10}" for s in specs)
    print(f"{name:<10}{row}")

讀表：`list` 是 `Sequence` 但不是 `Iterator`；`generator` 是 `Iterator` 但不能用 `[0]`；`int` 什麼都不是，所以 `sum(42)` 會炸。

所以函式簽名寫 `Iterable[int]` 比寫 `list[int]` 更誠實——它說的是「我需要的能力」：

In [ ]:
def total(nums: Iterable[int]) -> int:
    return sum(nums)


print(total([1, 2, 3]), total((1, 2, 3)), total({1, 2, 3}), total(n for n in range(4)))

### 3-1 自己寫規格：`abc.ABC` + `@abstractmethod`

規定「所有模型客戶端都必須有 `chat`」。沒實作就**建立物件時**直接報錯——比跑到一半才炸好查得多。

In [ ]:
from abc import ABC, abstractmethod


class ChatModel(ABC):
    @abstractmethod
    def chat(self, text: str) -> str: ...


class EchoModel(ChatModel):
    def chat(self, text: str) -> str:
        return f"你說：{text}"


class LazyModel(ChatModel):      # 忘了寫 chat
    pass


print(EchoModel().chat("嗨"))
for cls in (ChatModel, LazyModel):
    try:
        cls()
    except TypeError as exc:
        print(f"{cls.__name__}() 被擋下：{exc}")

---
## 4. 書和書籤：`Iterable` vs `Iterator`

<img src="images/iterable_vs_iterator.jpg" alt="Iterable 是書，Iterator 是書籤" width="560">

**白話**：`list` 是一本書，想翻幾次就翻幾次。`iter(書)` 會給你一張**書籤**，書籤只會往後走，走到底就沒了。

**專有名詞**

| | 正式說法 | 必備的 dunder | 能重來嗎 |
|---|---|---|---|
| 書 | **Iterable**（可迭代物件） | `__iter__` | 可以 |
| 書籤 | **Iterator**（迭代器） | `__iter__` + `__next__` | 不行，耗盡（exhausted）就空 |

`for x in 書` 背後其實是：先 `iter(書)` 拿書籤，再一直 `next(書籤)`，直到收到 `StopIteration`。

In [ ]:
book = [1, 2, 3]
bookmark = iter(book)

print("next ->", next(bookmark))
print("next ->", next(bookmark))
print("剩下 ->", list(bookmark))
print("再拿 ->", list(bookmark), " ← 書籤走到底了")
print("書本身 ->", book, " ← 書不受影響")

自己做一張書籤：寫出 `__iter__` 和 `__next__`，**不用繼承** `Iterator`，`isinstance` 照樣認——這就是「認能力，不認身分」。

In [ ]:
class CountDown:
    def __init__(self, n):
        self.n = n

    def __iter__(self):
        return self

    def __next__(self):
        if self.n == 0:
            raise StopIteration       # 告訴 for：沒有下一個了
        self.n -= 1
        return self.n + 1


cd = CountDown(3)
print(isinstance(cd, Iterator))   # True，沒繼承也算
print(list(cd))
print(list(cd), " ← 第二次就空了")

---
## 5. 做好一道上一道：`yield` 與 generator

<img src="images/yield_vs_return.jpg" alt="return 一次上齊，yield 做好一道上一道" width="560">

**白話**：`return` 是整桌菜一次端上來，端完廚房就關門。`yield` 是做好一道上一道，廚房**暫停**等你吃完，再接著做下一道。

**專有名詞**

- 函式裡只要有 `yield`，呼叫它不會執行內容，而是拿到一個 **generator**（產生器，`Iterator` 的一種）。
- 這種「要的時候才算」叫 **lazy evaluation**（惰性求值）。
- 課程的 `course_utils.llm.stream()` 回傳 `Iterator[str]`，模型生出一段就 `yield` 一段——所以聊天畫面會一個字一個字跳出來。

In [ ]:
import time
from collections.abc import Iterator


def stream() -> Iterator[str]:
    for piece in ["你", "好", "，", "我", "是", "模", "型", "！"]:
        time.sleep(0.15)          # 假裝模型正在算下一個 token
        yield piece


gen = stream()
print("呼叫後拿到：", gen, " ← 還沒開始跑")
for piece in gen:
    print(piece, end="", flush=True)
print()

`Iterator` 沒有長度、不能用編號。想要完整字串就全部接起來；要用第二次就重新呼叫：

In [ ]:
result = stream()
try:
    result[0]
except TypeError as exc:
    print("result[0] ->", exc)

print("".join(stream()))

---
## 6. 繼承是填空，組合是裝零件

<img src="images/inherit_vs_compose.jpg" alt="繼承是填空，組合是裝零件" width="560">

*圖：左邊表格的欄位內容只是示意，重點是「流程印好了，只留一格給你填」。*

**白話**

- **繼承**：框架印好一張公版表格（固定流程），你只填其中一格。框架會回頭呼叫你填的那格——所以你在自己的程式裡找不到誰呼叫 `_retrieve`、`_run`，是正常的。
- **組合**：機器本身不做事，它**拿著**可抽換的零件。測試時換一塊假零件就能跑。

**專有名詞**

| 白話 | 正式說法 | 關係 | 寫法 |
|---|---|---|---|
| 填公版表格 | **inheritance**（繼承）、**override**（覆寫）、template method 模式 | is-a（是一種） | `class A(B):` |
| 裝零件 | **composition**（組合） | has-a（擁有） | `self.b = b` |
| 零件從外面傳進來 | **dependency injection**（依賴注入） | | `__init__(self, b)` |

In [ ]:
class Model:
    """父類別：流程寫死，只留 _generate 給子類別填。"""

    def __init__(self, name):
        self.name = name

    def reply(self, text):
        return f"[{self.name}] {self._generate(text)}"   # 固定流程

    def _generate(self, text):
        raise NotImplementedError


class Echo(Model):
    def __init__(self):
        super().__init__("echo")      # 先跑父類別的出廠設定

    def _generate(self, text):        # 只填這一格
        return text


class Shout(Model):
    def __init__(self):
        super().__init__("shout")

    def _generate(self, text):
        return text.upper() + "!!!"


for m in (Echo(), Shout()):
    print(m.reply("hello"))           # 同一行呼叫，不同行為 = 多型 (polymorphism)

print("Echo 的繼承鏈：", [c.__name__ for c in Echo.__mro__])

In [ ]:
class RealDB:
    def status(self, service):
        return f"{service}: 查真正的資料庫……"


class FakeDB:
    def status(self, service):
        return f"{service}: healthy（假資料，測試用）"


class ReadTools:
    def __init__(self, operations):   # 零件從外面傳進來 = 依賴注入
        self.operations = operations

    def check(self, service):
        return self.operations.status(service)


print(ReadTools(RealDB()).check("order-api"))
print(ReadTools(FakeDB()).check("order-api"))   # 換零件，ReadTools 一行都不用改

---
## 7. 誰會檢查資料？`dataclass` / `TypedDict` / Pydantic

<img src="images/data_classes_guard.jpg" alt="三種資料類別：誰會檢查？" width="560">

**白話**：Agent 程式大半在搬資料。三種寫法長得很像，差別在**門口有沒有警衛**。

| 寫法 | 白話 | 執行時驗證 | 取值 | 課程用途 |
|---|---|---|---|---|
| `@dataclass` | 省掉手寫 `__init__` 的箱子 | 否 | `x.name` | 內部設定（`Endpoint`） |
| `TypedDict` | 貼了欄位標籤的 dict | 否 | `x["name"]` | LangGraph state |
| Pydantic `BaseModel` | 門口有警衛 | **是** | `x.name` | 工具參數、模型輸出 |

**判斷口訣**：資料來自外面（使用者、**模型**）→ 一律 Pydantic。模型給的工具參數可能亂寫，警衛要在執行前擋下。

In [ ]:
from dataclasses import FrozenInstanceError, dataclass
from typing import TypedDict


@dataclass(frozen=True)
class Endpoint:
    name: str
    limit: int


class State(TypedDict):
    name: str
    limit: int


ep = Endpoint("local", limit="999 個")        # 型別錯了也照收
st: State = {"name": "local", "limit": "999 個"}

print("dataclass ->", ep)
print("TypedDict ->", st, type(st))          # 就是普通 dict
try:
    ep.limit = 1
except FrozenInstanceError as exc:
    print("frozen=True 不准改 ->", exc)

In [ ]:
from typing import Literal

from pydantic import BaseModel, ConfigDict, Field, ValidationError


class LogArgs(BaseModel):
    model_config = ConfigDict(extra="forbid")           # 多給欄位就拒絕

    service: Literal["order-api", "payment-api"]        # 只准這兩個值
    limit: int = Field(default=10, ge=1, le=50)         # 範圍 1～50


print("正常 ->", LogArgs(service="order-api"))
print("字串 '5' 自動轉 int ->", LogArgs(service="order-api", limit="5"))

bad_calls = [
    {"service": "order-api", "limit": 999},
    {"service": "user-api"},
    {"service": "order-api", "rm_rf": True},
]
for args in bad_calls:
    try:
        LogArgs(**args)
    except ValidationError as exc:
        err = exc.errors()[0]
        print(f"擋下 {args} -> {err['loc']}: {err['msg']}")

順帶一提，`LogArgs(**args)` 的 `**` 是把 dict **攤開成具名參數**（keyword unpacking），等同 `LogArgs(service="order-api", limit=999)`。框架把模型吐出的 JSON 轉成工具參數時，就是這一招。

---
## 8. 把函式送去加工：`@` 裝飾器

<img src="images/decorator_station.jpg" alt="@ 裝飾器：把函式送去加工" width="560">

**白話**：`@tool` 放在函式上面，就像把函式放上輸送帶，經過一台加工機——加工機可以**幫它登記**、**幫它包一層功能**，再把結果放回原來的名字。

**專有名詞**

- **decorator**（裝飾器）：接收一個函式、回傳一個函式（或物件）的函式。
- `@deco` 寫在 `def f` 上方 ≡ 定義完後執行 **`f = deco(f)`**。
- **registry**（註冊表）：框架用來記「有哪些工具／路由」的清單。

看到 `@xxx` 就問一句：**它把我的函式登記到哪，或加了什麼功能？**

In [ ]:
registry = {}


def tool(func):
    registry[func.__name__] = func.__doc__     # 登記：名字 + 說明
    return func


@tool
def read_logs(service: str) -> str:
    """讀某個服務最近的日誌。"""
    return f"{service} 的日誌……"


@tool
def restart(service: str) -> str:
    """重啟某個服務（寫入操作，要先核准）。"""
    return f"{service} 已重啟"


for name, doc in registry.items():
    print(f"{name:<10} {doc}")

這就是 LangChain `@tool`、MCP `@mcp.tool` 的核心：**docstring 會變成給模型看的工具說明**。所以工具的 docstring 要寫給模型讀，不是寫給自己。

再看一個「包一層功能」的版本——幫函式加上計時，原函式一行都不改：

In [ ]:
import functools
import time


def timed(func):
    @functools.wraps(func)                      # 保留原函式的名字與 docstring
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f"  ⏱ {func.__name__} 花了 {time.perf_counter() - start:.3f}s")
        return result
    return wrapper


@timed
def slow_search(query):
    time.sleep(0.2)
    return f"找到 3 筆關於「{query}」的結果"


print(slow_search("timeout"))
print("名字還是：", slow_search.__name__)

---
## 讀碼速查：打開陌生類別，問四題

| # | 問題 | 看到什麼 → 直接判斷 |
|---|---|---|
| 1 | 括號裡繼承誰？ | `(BaseModel)` 會驗證的資料／`(TypedDict)` 有標籤的 dict／`(BaseXxx)` 框架填空／`(XxxError)` 自訂例外 |
| 2 | `__init__` 收了什麼？ | 它手上有哪些零件（組合） |
| 3 | 哪些方法沒有 `_`？ | 外面能按的按鈕（public API） |
| 4 | 方法上面有 `@` 嗎？ | 誰會來呼叫它（框架、路由、工具註冊） |

## 自我檢查

```python
class SearchTool(BaseTool):
    def __init__(self, index):
        super().__init__()
        self._index = index

    @property
    def size(self):
        return len(self._index)

    def _run(self, query):
        return self._index.search(query)
```

1. `SearchTool` 和 `BaseTool` 是什麼關係？和 `index` 呢？
2. `_run` 大概是誰在呼叫？
3. `len(self._index)` 背後呼叫哪個 dunder？
4. 同一個 generator 跑兩次 `for`，第二次看到什麼？
5. 工具參數要檢查範圍，用 `TypedDict` 還是 `BaseModel`？

<details>
<summary>答案</summary>

1. 繼承（is-a，填空）；組合（has-a，依賴注入）。
2. 框架——父類別的固定流程會回頭呼叫它。
3. `self._index.__len__()`。
4. 什麼都沒有，書籤第一次就走到底了。
5. `BaseModel`，`TypedDict` 執行時不驗證。

</details>